
# FEAT / FND free-energy postprocessing

This notebook is a replacement for the ODE/CNF `fes_odework.ipynb` postprocessing when inference was run with

```bash
--likelihood FND
```

It matches the **current** `latt` branch behavior where `model.inference(batch)` in FND mode returns **four objects**:

```python
logp, samples, aa_out, zs_minus_zs_mean
```

with

```python
logp = torch.concatenate([samples_logp, _samples_logp], dim=-1)
```

`FND-inference.py` then saves this as `Logp_<idx>.txt`, and saves the prior/Einstein reduced energy of the sampled initial point as `Uzs_<idx>.txt`.

For FEAT/FND, the log-probability terms are **path transition likelihoods**, not CNF log-Jacobians. The forward work is

\[
W_f = u_B(x_1)-u_A(x_0)+\log q^+(x_{0\to 1})-\log q^-(x_{1\to 0}\mid x_{0\to1}\text{ path}).
\]

The notebook therefore needs the target reduced energies `u_B(x_1)` of the generated final structures. The current `FND-inference.py` output does **not** save those target energies, so provide them as `target_uB_file` below, or add a small cell that evaluates your potential on the generated structures.


In [ ]:

from pathlib import Path
import re
import numpy as np
import pandas as pd
from scipy.special import logsumexp, expit
import matplotlib.pyplot as plt

# Folder containing FND-inference.py outputs:
#   Logp_<idx>.txt
#   Uzs_<idx>.txt
#   optionally generated structures / target energies
out_dir = Path("./data/")

# Temperature conversion for reduced free energy -> eV/atom.
# If your u = U/kBT, then ΔF[eV] = Δf_reduced * T_in_eV.
temperature_K = 1600
T_in_eV = 8.617333262145e-5 * temperature_K
crystal_type = 'quartz'
num_atoms = 1125

# Provide target reduced energies u_B(x_1) for the generated final structures.
# Supported formats:
# 1. None: notebook will load forward logp and prior energy, then stop before ΔF.
# 2. One text file with one u_B per rollout/sample.
# 3. A directory pattern handled in load_target_uB below.


In [ ]:
def _idx_from_name(path: Path) -> int:
    m = re.search(r"_(\d+)\.(?:txt|dat)$", path.name)
    if not m:
        raise ValueError(f"Cannot parse integer index from {path.name}")
    return int(m.group(1))


def as_1d(x):
    return np.asarray(x, dtype=float).reshape(-1)


def logmeanexp(x):
    x = as_1d(x)
    return logsumexp(x) - np.log(x.size)


def ess_from_logw(logw):
    logw = as_1d(logw)
    return np.exp(2.0 * logsumexp(logw) - logsumexp(2.0 * logw))


def split_saved_fnd_logp(path: Path):
    """
    Current FND-inference.py saves:
        np.savetxt(filename_logp, logp.view(1, -1).numpy())

    Current fed_wrapper.py returns:
        logp = concatenate([samples_logp, _samples_logp], dim=-1)

    This loader handles either a saved 2-column array or a flattened one-row array.

    Returns
    -------
    logq_plus, logq_minus : 1D arrays
        logq_plus is the forward sampling transition log-probability.
        logq_minus is the reverse/opposite transition log-probability evaluated on the same forward path.
    """
    arr = np.loadtxt(path)
    arr = np.asarray(arr, dtype=float)

    if arr.ndim == 0:
        raise ValueError(f"{path} contains only one scalar; expected two FND logp parts.")

    # Case A: saved as N x 2 or 1 x 2
    if arr.ndim == 2 and arr.shape[-1] == 2:
        return arr[..., 0].reshape(-1), arr[..., 1].reshape(-1)

    flat = arr.reshape(-1)
    if flat.size % 2 != 0:
        raise ValueError(f"{path} has {flat.size} values; cannot split into two equal FND logp parts.")

    half = flat.size // 2
    return flat[:half], flat[half:]


def load_uA_from_uzs(path: Path):
    """
    Current FND-inference.py writes one prior reduced energy in Uzs_<idx>.txt:
        [[ ((zs @ cell)**2 / (2*x0std**2)).sum() ]]

    Older EJE notebooks sometimes had more columns. Here we take the first column as u_A(x0).
    """
    arr = np.loadtxt(path)
    arr = np.asarray(arr, dtype=float)
    flat = arr.reshape(-1)
    if flat.size < 1:
        raise ValueError(f"{path} is empty")
    return float(flat[0])


def load_target_uB(target_uB_file):
    if target_uB_file is None:
        return None
    target_uB_file = Path(target_uB_file)
    vals = np.loadtxt(target_uB_file)/T_in_eV

    return vals



## Load the current FND-inference.py outputs

Required files:

- `Logp_<idx>.txt`, containing the flattened concatenation `[logq_plus, logq_minus]`.
- `Uzs_<idx>.txt`, whose first value is the prior/Einstein reduced energy `u_A(x0)`.

The current script does **not** save a separate reverse-path ensemble. Therefore this notebook computes the **forward FEAT/Jarzynski estimator** from the saved outputs. True two-sided FEAT/BAR requires backward paths started from true target samples, not merely a reverse rollout from generated endpoints.


In [ ]:

logp_files = sorted(out_dir.glob("Logp_*.txt"), key=_idx_from_name)
uzs_files = {_idx_from_name(p): p for p in out_dir.glob("Uzs_*.txt")}
uB_files = {_idx_from_name(p): p for p in out_dir.glob("all_energy_atoms_*.dat")}

if len(logp_files) == 0:
    raise FileNotFoundError(f"No Logp_*.txt files found in {out_dir.resolve()}")

rows = []
rows_eV = []
for p in logp_files:
    idx = _idx_from_name(p)
    if idx not in uzs_files:
        raise FileNotFoundError(f"Missing Uzs_{idx}.txt for {p.name}")

    logq_plus, logq_minus = split_saved_fnd_logp(p)
    uA = load_uA_from_uzs(uzs_files[idx])
    uB = load_target_uB(uB_files[idx])
    if logq_plus.size != logq_minus.size:
        raise RuntimeError("Internal split error")

    # Usually one rollout per file. If not, keep sub-index j.
    for j, (lp, lm) in enumerate(zip(logq_plus, logq_minus)):
        rows.append({
            "idx": idx,
            "subidx": j,
            "logq_plus": lp,
            "logq_minus": lm,
            "uA_x0": uA,
            "uB_x1": uB,
        })

        rows_eV.append({
            "idx": idx,
            "subidx": j,
            "logq_plus": lp * T_in_eV/num_atoms,
            "logq_minus": lm * T_in_eV/num_atoms,
            "uA_x0": uA * T_in_eV/num_atoms,
            "uB_x1": uB * T_in_eV/num_atoms,
        })
idx_min_uB = np.argmin(np.array([rows[i]['uB_x1'] for i in range(len(rows))]))
rows.pop(idx_min_uB)
rows_eV.pop(idx_min_uB)
df = pd.DataFrame(rows).sort_values(["idx", "subidx"]).reset_index(drop=True)
print("Loaded rows:", len(df))

df.head()


In [ ]:
df_eV = pd.DataFrame(rows_eV).sort_values(["idx", "subidx"]).reset_index(drop=True)
df_eV.head()

In [ ]:

df["W_fwd"] = (
    df["uB_x1"].to_numpy()
    - df["uA_x0"].to_numpy()
    + df["logq_plus"].to_numpy()
    - df["logq_minus"].to_numpy()
)
df.head()


In [ ]:
sigma = np.sqrt(T_in_eV)
f0_analytic = -0.5 * 3 * np.log(2*np.pi*sigma**2)*num_atoms
df["F_fwd"] = (
    df["W_fwd"].to_numpy()
    + f0_analytic
)
df.head()

In [ ]:

df_eV["W_fwd"] = (
    df["W_fwd"] * T_in_eV/num_atoms
)

df_eV["F_fwd"] = (
    df["F_fwd"] * T_in_eV/num_atoms
)
df_eV.head()


## FEAT forward estimator

For the saved forward FND paths,

\[
\Delta f = -\log\left\langle e^{-W_f}\right\rangle,
\qquad
W_f = u_B(x_1)-u_A(x_0)+\log q^+ - \log q^- .
\]

Here `u_A` and `u_B` must both be **reduced** energies, i.e. divided by `kBT`.


In [ ]:

if "W_fwd" not in df:
    raise RuntimeError("Cannot compute ΔF: set target_uB_file or fill df['uB_x1'] first.")

W_fwd = df["W_fwd"].to_numpy()
logw_fwd = -W_fwd

Delta_f_fwd = -logmeanexp(-W_fwd) + f0_analytic
ESS_fwd = ess_from_logw(logw_fwd)

summary = {
    "n_samples": len(W_fwd),
    "DeltaF_forward_eV_per_atom": Delta_f_fwd * T_in_eV / num_atoms,
    "F_forward_eV_per_atom": (Delta_f_fwd+ f0_analytic) * T_in_eV / num_atoms,
    "ESS_forward": ESS_fwd,
    "ESS_forward_fraction": ESS_fwd / len(W_fwd),
    "W_mean": float(np.mean(W_fwd)* T_in_eV / num_atoms),
    "W_std": float(np.std(W_fwd)* T_in_eV / num_atoms),
    "W_min": float(np.min(W_fwd)* T_in_eV / num_atoms),
    "W_max": float(np.max(W_fwd)* T_in_eV / num_atoms),
}

pd.Series(summary)


In [ ]:
ref_ex = np.loadtxt(f"/home/tuoping/odefed_mdgen/workdir_odefed_mdgen/data/SiO2/npt_1600K_1GPa/npt_quartz_dense/npt/thermo-lammps.dat")[100:,2]

ref_ex_c = np.loadtxt(f"/home/tuoping/odefed_mdgen/workdir_odefed_mdgen/data/SiO2/npt_1600K_1GPa/npt_coesite_dense/npt/thermo-lammps.dat")[100:,2]


In [ ]:

plt.figure(figsize=(5, 3.5))
plt.hist(df_eV["uB_x1"].to_numpy(), bins=100, alpha=0.8, density=True, label=f"{temperature_K} K (FM)")
plt.axvline(df_eV["uB_x1"].to_numpy().max(), ls='--', c='k')
plt.axvline(df_eV["uB_x1"].to_numpy().min(), ls='--', c='k')
plt.axvline(np.median(df_eV["uB_x1"].to_numpy()), ls='dotted', c='k')

_ = plt.hist(ref_ex/num_atoms, bins=100, alpha=0.5, color='r', density=True, label="Quartz 1600 K")
plt.axvline(ref_ex.max()/num_atoms, ls='--', c='r')
plt.axvline(ref_ex.min()/num_atoms, ls='--', c='r')


_ = plt.hist(ref_ex_c/864, bins=100, alpha=0.5, color='green', density=True, label="Coesite 1600 K")
plt.axvline(ref_ex_c.max()/864, ls='--', c='green')
plt.axvline(ref_ex_c.min()/864, ls='--', c='green')

plt.xlabel(r"$E_{x_1}$")
plt.ylabel("Fraction")
plt.legend()
plt.xlim(-326.9, -326.48)
plt.tight_layout()
plt.show()


In [ ]:

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.scatter(df_eV["uB_x1"].to_numpy(), df_eV['W_fwd'], alpha=0.8)
ax.set_xlabel(r"$E_{x_1}$")
ax.set_ylabel("FEAT forward work W")

ax.legend()
plt.tight_layout()
plt.show()



## Optional: non-equilibrium BAR if you later save true backward target-started paths

The current `FND-inference.py` output path does not save a true target-started backward ensemble. If you later generate one with initial samples from the target distribution \(x_1\sim p_B\), save its work values as `W_bwd`, where the same work convention is used:

\[
W_b = u_B(x_1)-u_A(x_0)+\log q^+ - \log q^- .
\]

Then use the BAR cell below.


In [ ]:

def jarzynski_forward(W_fwd):
    return -logmeanexp(-as_1d(W_fwd))


def jarzynski_backward(W_bwd):
    return logmeanexp(as_1d(W_bwd))


def noneq_bar(W_fwd, W_bwd, max_iter=500, tol=1e-12):
    """
    FEAT/Crooks/BAR estimator using the same work variable W on forward and backward paths.

    Δf = log( <phi(-W_b + C)>_b / <phi(W_f - C)>_f ) + C,
    phi(x)=1/(1+exp(x)).
    """
    W_fwd = as_1d(W_fwd)
    W_bwd = as_1d(W_bwd)
    C = 0.5 * (jarzynski_forward(W_fwd) + jarzynski_backward(W_bwd))

    for _ in range(max_iter):
        # phi(-W_b + C) = expit(W_b - C)
        num = np.mean(expit(W_bwd - C))
        # phi(W_f - C) = expit(C - W_f)
        den = np.mean(expit(C - W_fwd))
        C_new = np.log(num) - np.log(den) + C
        if abs(C_new - C) < tol:
            C = C_new
            break
        C = C_new
    return C

# Example usage after constructing W_bwd:
# Delta_f_BAR = noneq_bar(W_fwd, W_bwd)
# print(Delta_f_BAR, Delta_f_BAR * T_in_eV / num_atoms)



## Sanity checks

Large variance or ESS close to 1 usually means the endpoint energy term or the path likelihood ratio is dominated by a few samples.


In [ ]:

cols = ["uA_x0", "logq_plus", "logq_minus"]
if "uB_x1" in df:
    cols += ["uB_x1", "W_fwd"]

df[cols].describe().T


In [ ]:

if "W_fwd" in df:
    # Contributions to W = uB - uA + logq_plus - logq_minus
    contrib = pd.DataFrame({
        "uB_minus_uA": df["uB_x1"].to_numpy() - df["uA_x0"].to_numpy(),
        "logq_plus_minus_logq_minus": df["logq_plus"].to_numpy() - df["logq_minus"].to_numpy(),
        "W_fwd": df["W_fwd"].to_numpy(),
    })
    display(contrib.describe().T)

    ax = contrib[["uB_minus_uA", "logq_plus_minus_logq_minus", "W_fwd"]].plot.hist(
        bins=50, alpha=0.5, figsize=(6,4)
    )
    ax.set_xlabel("reduced units")
    plt.tight_layout()
    plt.show()
